## Tutorial 6. ML force fields to simulate a realistic Pt–OH–H2O interface


## The goal of the notebook

This notebook demonstrates how a realistic catalytic interface containing a metal surface, adsorbate, and solvent molecules can be modeled and relaxed using the SpookyNet machine-learning force field.

## Description of the method

SpookyNet is a machine-learning force field trained on DFT data that predicts energies and forces while accounting for electronic effects such as polarization and charge transfer. It enables efficient simulations of catalytic interfaces that are difficult to treat with classical force fields.

SpookyNet github: https://github.com/OUnke/SpookyNet

SpookyNet article: Unke, O. T.; Chmiela, S.; Gastegger, M.; Sauceda, H. E.; Müller, K.-R.; Tkatchenko, A. SpookyNet: Learning Force Fields with Electronic Degrees of Freedom and Nonlocal Effects. Nat. Commun. 2021, 12, 7273. https://doi.org/10.1038/s41467-021-27504-0

## Outline of the workflow

STEP 1. Build a Pt(111) surface with an OH adsorbate and water molecules.

STEP 2. Run geometry optimization using Spookynet.

 Installation of packages

First thing first, we need to install the packages and import the neccessary functions. We need ASE,torch and Spookynet


In [ ]:
!git clone https://github.com/OUnke/SpookyNet.git
%cd SpookyNet
!pip install .

Next we import the necessary packages:

ASE https://ase-lib.org/


SpookyNet: https://github.com/OUnke/SpookyNet

Torch: https://github.com/pytorch/pytorch

In [ ]:
##-ASE
from ase import Atoms, Atom
from ase.constraints import FixAtoms
from ase.io import read, write, Trajectory
from ase.md import MDLogger
from ase.md.langevin import Langevin
from ase.units import fs
from ase.visualize import view
from ase.optimize import BFGS

##-General
import torch
import os

##-matplotlib
import matplotlib.pyplot as plt

##-Spookynet
from spookynet.spookynet_calculator import SpookyNetCalculator

## STEP 1. Preparing the Pt–OH–H₂O interface

In this step we load a prepared Pt(111)–OH–H₂O structure and create an ASE atoms object. The Pt atoms representing the metal slab are constrained to mimic a bulk surface, while the adsorbate and water molecules remain free to relax.

In [ ]:

##-Downloading the Pt-OH-H2O structure
!wget https://github.com/doublelayer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj

In [ ]:
##-Creating the atoms object
atoms = read(f'Pt-OH-H2O.traj')
view(atoms, viewer='x3d')

In [ ]:
##-Adding tag = 1 for each Pt atom
for atom in atoms:
    if atom.symbol == 'Pt':
        atom.tag = 1

##-Constraining atoms with tag = 1
indices = [atom.index for atom in atoms if atom.tag == 1]
atoms.set_constraint(FixAtoms(indices))


## STEP 2. Geometry optimization with Spookynet

The pretrained SpookyNet model is loaded and assigned as the calculator. It predicts energies and forces for the atomic configuration, replacing traditional DFT calculations in the optimization workflow. Using the predicted forces, a BFGS optimizer iteratively updates atomic positions until the forces fall below the convergence threshold, yielding a relaxed interface structure.

In [ ]:
atoms.center(vacuum=13.0, axis=2)
atoms.set_pbc(True)

calc = SpookyNetCalculator(
    load_from="parameters.pth",
    charge=0,
    magmom=0
)
atoms.calc = calc

output_dir = "data/Pt_OH_H2O_relax"
os.makedirs(output_dir, exist_ok=True)

output_traj = os.path.join(output_dir, "Pt-OH-H2O_optim.traj")

opt = BFGS(atoms, trajectory=output_traj)

opt.run(fmax=0.2, steps=1000)

/usr/local/lib/python3.12/dist-packages/ase/calculators/calculator.py:622: FutureWarning: The keyword "ignore_bad_restart_file" is deprecated and will be removed in a future version of ASE.  Passing more than one positional argument to Calculator is also deprecated and will stop functioning in the future.  Please pass arguments by keyword (key=value) except optionally the "restart" keyword.
  warnings.warn(
/content/SpookyNet/spookynet/spookynet_calculator.py:165: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  else torch.tensor([atoms.cell], dtype=self.dtype),


      Step     Time          Energy          fmax
BFGS:    0 08:22:33     -722.818970        4.930598
BFGS:    1 08:22:34     -731.257568        4.350372
BFGS:    2 08:22:36     -745.745300        2.718250
BFGS:    3 08:22:37     -750.257019        1.938689
BFGS:    4 08:22:39     -751.604797        0.899410
BFGS:    5 08:22:40     -752.206482        0.757131
BFGS:    6 08:22:41     -752.643005        0.581004
BFGS:    7 08:22:42     -752.799377        0.598486
BFGS:    8 08:22:42     -752.955200        0.377703
BFGS:    9 08:22:43     -753.043396        0.304695
BFGS:   10 08:22:43     -753.198303        0.379524
BFGS:   11 08:22:44     -753.327148        0.384026
BFGS:   12 08:22:45     -753.414062        0.350373
BFGS:   13 08:22:45     -753.456726        0.192510
BFGS:   14 08:22:46     -753.482910        0.214424
BFGS:   15 08:22:46     -753.513794        0.223408
BFGS:   16 08:22:46     -753.568481        0.255138
BFGS:   17 08:22:46     -753.661926        0.366056
BFGS:   18 08:

np.False_

In [ ]:
atoms = read('/content/SpookyNet/data/Pt_OH_H2O_relax/Pt-OH-H2O_optim.traj', index=-1)
view(atoms, viewer='x3d')

## Conclusive remarks

In this tutorial, we started from a Pt(111) surface model containing an OH adsorbate and surrounding water molecules to represent a realistic catalytic interface. The structure was loaded into ASE and prepared for simulation by constraining the platinum atoms of the slab to mimic a bulk surface environment. Next, the SpookyNet machine-learning force field was attached as the calculator to predict energies and forces for the system. Using these ML-predicted forces, the interface was relaxed with a BFGS optimizer until the atomic configuration reached a stable geometry. This workflow demonstrates how ML force fields can be integrated into atomistic simulations of catalytic interfaces containing solvent molecules.

After completing this tutorial, you will be able to:

Apply structural constraints to represent a realistic surface environment during simulations.

Run geometry optimizations using a machine-learning force field through the SpookyNet framework.

Integrate ML-based force fields into an ASE workflow for efficient relaxation of catalytic interfaces.